In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
#Hfus dan Tm masukin ke csv yh jgn lupa

like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
leucine,131.17,6.9327,2.9136,297.3468,2,2
water,18.02,1.2047,2.801457,353.94,1,1
"""

unlike_parameter = """Clapeyron Database File
PCSAFT Unlike Parameters [csvtype = unlike,grouptype = PCSAFT]
species1,species2,k
water,leucine,-0.063
"""

#jgn lupa combing rules yh
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
water,H,water,e,2425.67,0.045
leucine,H,leucine,e,3600,0.02
water,H,leucine,e,3012.835,0.029994898
water,e,leucine,H,3012.835,0.029994898
"""
components = ["water", "leucine"]
model = CompositeModel(components;
                       fluid = PCSAFT,
                       solid = SolidHfus,
                       fluid_userlocations = [like_parameter, unlike_parameter, assoc_parameter])

println(model.fluid.params.epsilon.values)
println(model.fluid.params.sigma.values)
println("======================")
println(model.fluid.params.epsilon_assoc.values)
println(model.fluid.params.bondvol.values)
println("kij = ", (1  - ((model.fluid.params.epsilon.values[2])/(sqrt(model.fluid.params.epsilon.values[1] * model.fluid.params.epsilon.values[4])))))

[353.94 344.84959662473415; 344.84959662473415 297.3468]
[2.8014570000000003e-10 2.8575285000000005e-10; 2.8575285000000005e-10 2.9136e-10]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 3012.835, 3012.835, 3600.0]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.029994898, 0.029994898, 0.02]
kij = -0.06299999999999994


In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("gamma_leucine.csv")
fix_line_endings("rhol_leucine.csv")

Fixed: gamma_leucine.csv
Fixed: rhol_leucine.csv


In [4]:
#SOLUTION DENSITY

Mw_water = model.fluid.params.Mw[1] / 1000
Mw_aa = model.fluid.params.Mw[2] / 1000

println("MW Solute  : ", Mw_aa, " kg/m3")
println("MW solvent : ", Mw_water, " kg/m3")

function molality_to_z(m::Float64, Mw_solute::Float64)
    # m mol solute per kg water
    n_solute = m
    n_water  = 1.0 / Mw_water   # mol water per kg water = 55.51
    x_water  = n_water / (n_water + n_solute)
    x_solute = n_solute / (n_water + n_solute)
    return [x_water, x_solute]   # [water, amino acid]
end

function solution_density(model::EoSModel, m::Float64)
    z   = molality_to_z(m, Mw_aa)
    T   = 298.15
    p   = 101325.0
    V   = volume(model, p, T, z; phase=:liquid)        # m³/mol mixture
    Mw_mix = z[1]*Mw_water + z[2]*Mw_aa               # kg/mol
    return Mw_mix / V                                   # kg/m³
end

MW Solute  : 0.13116999999999998 kg/m3
MW solvent : 0.01802 kg/m3


solution_density (generic function with 1 method)

In [5]:
#ACTIVITY COEFFICIENT

function chem_activity_coeff(model::EoSModel, m::Float64)
    z     = molality_to_z(m, 75.07)
    z_inf = molality_to_z(1e-10, 75.07)   # very dilute reference
    p     = 101325.0
    T     = 298.15
    RT    = Rgas(model) * T

    # get chemical potentials directly
    chem_pot     = chemical_potential(model.fluid, p, T, z;     phase=:liquid)
    chem_pot_inf = chemical_potential(model.fluid, p, T, z_inf; phase=:liquid)

    # γ dari persamaan chemical potential
    gamma_star_x = exp((chem_pot[2] - chem_pot_inf[2]) / RT) * (z_inf[2] / z[2])

    # convert to molality basis (Kuramochi eq 5)
    #println("exp((μ[2] - μ_inf[2]) / RT) :", exp((μ[2] - μ_inf[2]) / RT))
    #println("m :",m)
    #println("z_inf :",z_inf)
    #println("z[2]: ",z[2])
    #println("z_inf[2] / z[2] :",z_inf[2] / z[2])
    return gamma_star_x / (1.0 + Mw_water * m)
end

chem_activity_coeff (generic function with 1 method)

In [8]:
toestimate = [
    Dict(
        :param => :epsilon,
        :indices => (1,2),
        :lower => 300.,
        :upper => 500.,
        :guess => 400.
    ),
    #Dict(
     #   :param => :sigma,
      #  :indices => (2,2),
       # :factor => 1e-10,
        #:lower => 2.0,
        #:upper => 8.0,
        #:guess => 2.7
    #),
    Dict(
        :param   => :epsilon_assoc,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 1000.0,
        :upper   => 5000.0,
        :guess   => 3500.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 0.01,
        :upper   => 0.05,
        :guess   => 0.025
    ),
]

3-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 500.0, :param => :epsilon, :indices => (1, 2), :guess => 400.0, :lower => 300.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => 4, :guess => 3500.0, :lower => 1000.0)
 Dict(:upper => 0.05, :param => :bondvol, :indices => 4, :guess => 0.025, :lower => 0.01)

In [9]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_leucine.csv",
        "gamma_leucine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))
println(x0)

Initial objective value: 0.40205879785458515
[400.0, 3500.0, 0.025]


In [10]:
method = ECA(; options = Options(iterations = 10000, seed = 42))
 
params_opt, model_opt = optimize(objective, estimator, method)

([353.7438175838358, 4002.6497856116903, 0.03653345223581984], CompositeModel{PCSAFT{BasicIdeal, Float64}, SolidHfus}("water", "leucine"))

In [11]:
println("\n=== Optimized PC-SAFT parameters for glycine ===")
println("  segment (m)  : ", model_opt.fluid.params.segment[2])
println("  sigma   (m)  : ", model_opt.fluid.params.sigma[2,2])
println("  epsilon (K1)  : ", model_opt.fluid.params.epsilon.values)
println("  epsilon_assoc  : ", model_opt.fluid.params.epsilon_assoc)
println("  bondvol  : ", model_opt.fluid.params.bondvol)
println("==================")
println(model_opt.fluid.params.epsilon.values)
println(model_opt.fluid.params.sigma.values)
println("kij = ", (1  - ((model_opt.fluid.params.epsilon.values[2])/(sqrt(model_opt.fluid.params.epsilon.values[1] * model_opt.fluid.params.epsilon.values[4])))))
println("======================")
println(model_opt.fluid.params.epsilon_assoc.values)
println(model_opt.fluid.params.bondvol.values)



=== Optimized PC-SAFT parameters for glycine ===
  segment (m)  : 6.9327
  sigma   (m)  : 2.9136e-10
  epsilon (K1)  : [353.94 353.7438175838358; 353.7438175838358 297.3468]
  epsilon_assoc  : AssocParam{Float64}("epsilon_assoc")[2425.67, 3012.835, 3012.835, 4002.6497856116903]
  bondvol  : AssocParam{Float64}("bondvol")[0.045, 0.029994898, 0.029994898, 0.03653345223581984]
[353.94 353.7438175838358; 353.7438175838358 297.3468]
[2.8014570000000003e-10 2.8575285000000005e-10; 2.8575285000000005e-10 2.9136e-10]
kij = -0.09041646495185995
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 3012.835, 3012.835, 4002.6497856116903]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.029994898, 0.029994898, 0.03653345223581984]


In [12]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [13]:
aard_p   = calculate_AAD(model_opt, "rhol_leucine.csv", solution_density)


=== AAD: rhol_leucine.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
0.0312      1000.757000   993.023540    0.7728  
0.0429      1001.037000   993.204885    0.7824  
0.0450      1001.078000   993.238248    0.7831  
0.0522      1001.264000   993.349402    0.7905  
0.0606      1001.464000   993.479728    0.7973  
0.0611      1001.488000   993.487614    0.7988  
0.0703      1001.716000   993.629743    0.8072  
0.0763      1001.832000   993.722600    0.8095  
0.0805      1001.958000   993.786701    0.8155  
0.0810      1001.990000   993.794864    0.8179  
0.0862      1002.085000   993.875059    0.8193  
AARD = 0.7995%


0.7994786726890183

In [14]:
aard_p   = calculate_AAD(model_opt, "gamma_leucine.csv", chem_activity_coeff)


=== AAD: gamma_leucine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.0400      1.015700      1.015173      0.0519  
0.0800      1.031500      1.030846      0.0634  
0.1200      1.047200      1.046987      0.0203  
0.1600      1.062900      1.063570      0.0630  
AARD = 0.0497%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.049667867252105875